# Explore Buildings Data

Do the inspection lat/lon coords line up with buildings?

In [2]:
import os
import sys
import folium
import numpy as np
import pandas as pd
import geopandas as gpd

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as colors

sys.path.append("../utils")

import config

### Import SB buildings data

In [3]:
sb_buildings_path = os.path.join(
    config.data_dir, "microsoft_buildings", "sb_buildings.geojson"
)

buildings = gpd.read_file(sb_buildings_path)

### Check whether inspection points line up with buildings

#### Import inspections

In [4]:
# Function to read in inspections data for 2019-2023
def import_inspections_data(year):
    """
    Function to read in inspections data for a given year and convert to Albers CRS.
    """
    inspections = os.path.join(
        config.cleaned_inspections_dir, f"inspections_{year}.geojson"
    )
    inspections = gpd.read_file(inspections).to_crs(config.albers_crs)
    return inspections


# Read in inspection data for 2019-2023 as separate dataframes
inspections_dict = {}
for year in range(2019, 2024):
    inspections_dict[year] = import_inspections_data(year)

In [5]:
# Unpack from dictionary
inspections_2019 = inspections_dict[2019]
inspections_2020 = inspections_dict[2020]
inspections_2021 = inspections_dict[2021]
inspections_2022 = inspections_dict[2022]
inspections_2023 = inspections_dict[2023]

#### Count number of inspections that fall outside of buildings

In [ ]:
# Create a function that returns the number of inspections that fall outside building geometries for a given year
def inspections_within_buildings(year):
    """
    Function to return the number of inspections that fall within building geometries for a given year.
    """
    inspections = import_inspections_data(year)
    intersects = inspections.sjoin(buildings, predicate="intersects")
    
    return intersects, inspections.loc[~inspections.index.isin(intersects.index)]

# Get the number of inspections that fall within building geometries for each year
inspections_2019, inspections_2019_buildings_false = inspections_within_buildings(2019)
inspections_2020, inspections_2020_buildings_false = inspections_within_buildings(2020)
inspections_2021, inspections_2021_buildings_false = inspections_within_buildings(2021)
inspections_2022, inspections_2022_buildings_false = inspections_within_buildings(2022)
inspections_2023, inspections_2023_buildings_false = inspections_within_buildings(2023)


In [ ]:
# For loop to print number of inspections that fall within and outside building geometries for each year
for year in range(2019, 2024):
    inspections_within, inspections_outside = inspections_within_buildings(year)
    print(f"{year} Inspections: \nWithin Buildings - {inspections_within.shape[0]} \nOutside of Buildings - {inspections_outside.shape[0]}\n")

2019 Inspections: 
Within Buildings - 11011 
Outside of Buildings - 3664

2020 Inspections: 
Within Buildings - 9528 
Outside of Buildings - 2350

2021 Inspections: 
Within Buildings - 10491 
Outside of Buildings - 3521

2022 Inspections: 
Within Buildings - 10157 
Outside of Buildings - 3485

2023 Inspections: 
Within Buildings - 9919 
Outside of Buildings - 3454



#### Count number of non-compliant inspections that fall outside of buildings

In [27]:
# For loop to print number of non-compliant inspections that fall within and outside building geometries for each year
for year in range(2019, 2024):
    non_comp_inspections_in, non_comp_inspections_out = inspections_within_buildings(year)
    non_comp_inspections_in = non_comp_inspections_in[non_comp_inspections_in["status"] == "Non-Compliant"]
    non_comp_inspections_out = non_comp_inspections_out[non_comp_inspections_out["status"] == "Non-Compliant"]
    print(f"{year} Non-Compliant Inspections: \nWithin Buildings - {non_comp_inspections_in.shape[0]} \nOutside of Buildings - {non_comp_inspections_out.shape[0]}\n")

2019 Non-Compliant Inspections: 
Within Buildings - 72 
Outside of Buildings - 51

2020 Non-Compliant Inspections: 
Within Buildings - 74 
Outside of Buildings - 20

2021 Non-Compliant Inspections: 
Within Buildings - 63 
Outside of Buildings - 7

2022 Non-Compliant Inspections: 
Within Buildings - 47 
Outside of Buildings - 13

2023 Non-Compliant Inspections: 
Within Buildings - 54 
Outside of Buildings - 17



### Calculate distance between non-compliant inspections outside of buildings and nearest building

In [28]:
from scipy.spatial import cKDTree
from shapely.geometry import Point

gpd1 = non_comp_inspections_in
gpd2 = non_comp_inspections_out

def ckdnearest(gdA, gdB):

    nA = np.array(list(gdA.geometry.apply(lambda x: (x.x, x.y))))
    nB = np.array(list(gdB.geometry.apply(lambda x: (x.x, x.y))))
    btree = cKDTree(nB)
    dist, idx = btree.query(nA, k=1)
    gdB_nearest = gdB.iloc[idx].drop(columns="geometry").reset_index(drop=True)
    gdf = pd.concat(
        [
            gdA.reset_index(drop=True),
            gdB_nearest,
            pd.Series(dist, name='dist')
        ], 
        axis=1)

    return gdf

ckdnearest(gpd1, gpd2)


,fulcrum_id,created_at,updated_at,created_by,updated_by,system_created_at,system_updated_at,version,status,project,...,previousyrinspecteddata,previousyrinspectionstatus,calculatededitor,calculatededitdate,creationdate,editdate,creator,editor,Date,dist
0,cb6c5509-0cf2-4c52-af41-93548c835f79,2022-04-04 17:00:00,2023-06-13 17:00:00,gis@sbcfire.com,gis@sbcfire.com,2022-04-05 13:47:20,2023-07-13 08:12:55,7,Non-Compliant,None,...,None,None,None,None,2022-07-05,2022-07-05,None,None,2023-06-14,268.067978
1,a85e65ea-3bed-4a4c-88c0-e53a843ee801,2022-04-04 17:00:00,2023-09-02 13:37:34,gis@sbcfire.com,burnpermits@sbcfire.tech,2022-04-05 13:48:00,2023-09-02 13:37:36,8,Non-Compliant,None,...,None,None,None,None,2022-08-21,2020-06-15,None,None,2022-08-21,11908.004645
2,ad4a7f44-cd86-4f7c-aa5d-47035ee388a4,2022-04-04 17:00:00,2023-10-11 10:39:57,gis@sbcfire.com,sbc.dsp.inspector@sbcfire.com,2022-04-05 13:49:33,2023-10-11 10:40:54,5,Non-Compliant,None,...,None,None,None,None,2022-10-03,2020-07-27,None,None,2023-09-08,8085.444748
3,063f1f26-dd68-414b-9e24-7a85c277089d,2022-04-04 17:00:00,2023-06-05 17:00:00,gis@sbcfire.com,gis@sbcfire.com,2022-04-05 13:50:10,2023-07-13 08:15:30,7,Non-Compliant,None,...,None,None,None,None,2022-10-22,2020-11-03,None,None,2023-06-21,6044.189590
4,cb489b62-081f-4435-8518-d34c955e8894,2022-04-04 17:00:00,2023-08-14 11:44:35,gis@sbcfire.com,sbc.dsp.inspector@sbcfire.com,2022-04-05 13:50:59,2023-08-15 08:30:55,5,Non-Compliant,None,...,None,None,None,None,2022-10-03,2020-07-27,None,None,2023-09-08,6280.866764
5,7f2e0fda-7978-49ec-92ad-70b7510bcc2c,2022-04-04 17:00:00,2023-08-10 11:28:23,gis@sbcfire.com,sbc.dsp.inspector@sbcfire.com,2022-04-05 13:52:23,2023-08-10 11:33:04,7,Non-Compliant,None,...,None,None,None,None,2022-10-03,2020-07-27,None,None,2023-09-08,3779.877258
6,8a9ae782-884b-4e87-9f72-db5883c46dd3,2022-04-04 17:00:00,2023-08-14 10:44:33,gis@sbcfire.com,sbc.dsp.inspector@sbcfire.com,2022-04-05 13:52:36,2023-08-14 11:30:03,7,Non-Compliant,None,...,None,None,None,None,2022-10-03,2020-07-27,None,None,2023-09-08,8402.594316
7,6086fd6e-0ff8-4673-8f48-c4e724437c66,2022-04-04 17:00:00,2023-09-05 11:31:45,gis@sbcfire.com,sbc.dsp.inspector@sbcfire.com,2022-04-05 13:56:17,2023-09-06 10:00:00,9,Non-Compliant,None,...,None,None,None,None,2023-07-26,2020-06-16,None,None,2023-07-25,169.487085
8,cd561c70-11d7-4dcd-9500-7a37caf40608,2018-05-23 17:00:00,2023-09-08 12:08:17,gis@sbcfire.com,sbc.dsp.inspector@sbcfire.com,2022-04-06 09:55:25,2023-09-11 07:47:25,4,Non-Compliant,None,...,None,None,None,None,2022-10-06,2020-07-21,None,None,2023-09-08,120.140463
9,eaaa650f-d864-44ac-a0ca-500c5033cb98,2018-11-28 16:00:00,2023-09-08 11:53:26,gis@sbcfire.com,sbc.dsp.inspector@sbcfire.com,2022-04-06 09:56:32,2023-09-11 07:47:19,4,Non-Compliant,None,...,None,None,None,None,2022-10-06,2020-07-21,None,None,2023-09-08,106.544061


## *Create interactive map

***CAUTION: Exercise judgement before running!! Uses too much memory currently.**

In [ ]:
# # Create an interactive map
# m = buildings.explore(
#     color="blue",
#     name="Buildings",
#     tooltip=False,
#     style_kwds={"fillOpacity": 0.2, "weight": 0.5},
#     min_zoom=15, 
#     tiles="CartoDB positron",  # Lightweight basemap
# )

# # Use a subset of columns for tooltips
# tooltip_columns = ["status", "address_fu"] 

# # Add internal inspections
# m = intersects.explore(
#     m=m,
#     color="green",
#     marker_kwds={"radius": 1},
#     name="Inspections inside buildings",
#     tooltip=tooltip_columns, 
# )

# # # Add external inspections
# # m = inspections_2019_buildings_false.explore(
# #     m=m,
# #     color="red",
# #     marker_kwds={"radius": 1},
# #     name="Inspections outside buildings",
# #     tooltip=tooltip_columns, 
# # )

# # Add layer control
# folium.LayerControl().add_to(m)

# # Display the map
# m